# 1. Dependencies

## Common Libraries

In [3]:
import os
import glob
import time
from typing import cast, Any

In [4]:
import json
import joblib
from joblib import Parallel, delayed, parallel_config
from joblib import parallel_config

In [5]:
import math
import numpy as np
import pandas as pd
import polars as pl
import polars.selectors as cs

## Plotting

In [6]:
import matplotlib.pyplot as plt

## Pre-Processing

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import IncrementalPCA
from imblearn.over_sampling import SMOTE

## Models

### KNN

In [8]:
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors

## Evaluation

In [9]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    precision_recall_fscore_support,
    confusion_matrix
)

## Global Variables

### Path

In [10]:
PATH_CACHE = "cache"

In [11]:
PATH_FOLDER_CSV = "cse-cic-ids2018"
PATH_FOLDER_RAW = "data-raw"

In [12]:
PATH_FOLDER_NO_INF = "data-no-inf"

In [13]:
PATH_FOLDER_SPLIT_DATA = "data-split-feature"
PATH_FOLDER_SPLIT_LABEL = "data-split-label"

In [14]:
PATH_IMPUTER_TRANSFORMER = "median-imputer.json"
PATH_IMPUTER = os.path.join(PATH_CACHE,PATH_IMPUTER_TRANSFORMER)

In [15]:
PATH_FOLDER_IMPUTED = "data-imputed"

In [16]:
PATH_SCALER_TRANSFORMER = "standard-scaler.pkl"
PATH_SCALER = os.path.join(PATH_CACHE,PATH_SCALER_TRANSFORMER)

In [17]:
PATH_FOLDER_SCALED = "data-scaled"

In [18]:
PATH_FOLDER_IPCA_TRANSFORMER = os.path.join(PATH_CACHE,"pca")
PATH_IPCA = "ipca-transformer.pkl"

In [19]:
PATH_FOLDER_IPCA = "data-ipca"

In [20]:
PATH_FOLDER_SMOTE_TRANSFORMER = os.path.join(PATH_CACHE, "smote")
PATH_SMOTE = "smote-transformer.pkl"

In [21]:
PATH_FOLDER_SMOTE = "data-smote"

In [22]:
PATH_FOLDER_MODEL = "trained-model"

### Others

In [23]:
LABEL_COLUMN = "Label"

## Helper Functions

In [24]:
def get_all_file_names(source_folder_name:str,file_format:str = ""):
    """List file names in source_folder_name that end with `.{file_format}`, sorted alphabetically.

    Note: if file_format is left empty, this returns an empty list (files are only ever
    appended when a format is given), so a format should always be supplied.
    """
    files = []
    for file in os.listdir(source_folder_name):
        if len(file_format)>0:
            if file.endswith(f".{file_format}"):
                files.append(file)
    return sorted(files)


In [25]:
def filter_file_names(list_file_names:list, filter_word:str):
    """Return only the file names that contain filter_word as a substring (e.g. a date like "2018-02-14")."""
    filtered_file_names = []
    for file_name in list_file_names:
        if filter_word in file_name:
            filtered_file_names.append(file_name)
    return filtered_file_names

In [26]:
def to_pandas_dataframe(data, template_column: list) -> pd.DataFrame:
    """Build a DataFrame from `data` and reindex its columns to match `template_column`."""
    df = pd.DataFrame(data)
    df = df.reindex(columns=template_column)
    return df

In [27]:
def get_split_parquet_files(source_folder_name: str, split_name: str) -> list[str]:
    """Return sorted paths to all Parquet files under `source_folder_name/split_name` (e.g. .../train)."""
    files = sorted(glob.glob(os.path.join(source_folder_name, split_name, "*.parquet")))
    if not files:
        raise FileNotFoundError(f"No Parquet files found in {os.path.join(source_folder_name, split_name)}")
    print(f"Found {len(files)} {split_name} files")
    return files

In [28]:
def get_polars_data_frame_with_label(
        source_folder_name: str, 
        split_name: str = "train", 
        label_column: str = LABEL_COLUMN, 
        downcast_to_float32: bool = True
    ) -> tuple[pl.DataFrame, pl.Series]:

    label_column = label_column.lower()
    files = get_split_parquet_files(source_folder_name, split_name)

    lazy_frame = pl.concat(pl.scan_parquet(file) for file in files)
    if downcast_to_float32:
        lazy_frame = lazy_frame.with_columns(cs.numeric().cast(pl.Float32))

    dataframe = lazy_frame.collect(engine="streaming")
    if label_column not in dataframe.columns:
        raise ValueError(f"Label column '{label_column}' not found in dataframe.")
    x = dataframe.drop(label_column)
    y = dataframe[label_column]
    return x, y

In [29]:
def get_polars_data_frame_without_label(
        source_folder_name: str, 
        split_name: str="", 
        downcast_to_float32: bool = True
    ) -> pl.DataFrame:
    if len(split_name) > 0:
        files = get_split_parquet_files(source_folder_name, split_name)
    else:
        files = get_all_file_names(source_folder_name)
    total = len(files)

    lazy_frames = []
    for i,file in enumerate(files,start=1):
        print(f"Processing [{i}/{total}] {file}...", end="\r", flush=True)
        lazy_frames.append(pl.scan_parquet(file))
    lazy_frame = pl.concat(lazy_frames, how="diagonal_relaxed")

    if downcast_to_float32:
        lazy_frame = lazy_frame.with_columns(cs.numeric().cast(pl.Float32))

    dataframe = lazy_frame.collect(engine="streaming")

    # del lazy_frames, lazy_frame, files, total
    return dataframe

### Model Training

In [30]:
def check_openmp_threads() -> int:
    """Report the thread count that actually governs the KNN search.

    Once sklearn dispatches a brute-force search to `ArgKmin.compute()` it
    threads with OpenMP and ignores `n_jobs` entirely. If this prints 1 the
    search runs single-threaded and will take roughly `n_cores` times longer.
    """
    from sklearn.utils._openmp_helpers import _openmp_effective_n_threads

    n_threads = _openmp_effective_n_threads()
    print(f"OpenMP effective threads: {n_threads}")
    try:
        import threadpoolctl
        for info in threadpoolctl.threadpool_info():
            print(f"  {info['user_api']:>8} / {info['internal_api']:<12}"
                  f" threads={info['num_threads']}")
    except ImportError:
        print("  (pip install threadpoolctl for per-library detail)")
    return n_threads

In [31]:
def dump_trained_model(model,name: str,subfolder: str,folder: str = PATH_FOLDER_MODEL):
    target_folder = os.path.join(folder, subfolder)
    os.makedirs(target_folder, exist_ok=True)
    file_path = os.path.join(target_folder,name)
    joblib.dump(model, file_path)
    return file_path

# 2. Pre-Processing

## 2.1. Change CSV to Parquet

In [ ]:
CHUNK_SIZE = 100000
COLUMNS_TO_DROP = {
    "flow id", "src ip", "source ip", "src port", "source port",
    "dst ip", "destination ip", "timestamp",
}

In [ ]:
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df.columns = df.columns.str.strip().str.lower()
    return df

In [ ]:
def drop_unwanted_columns(df: pd.DataFrame) -> pd.DataFrame:
    return df.drop(columns=COLUMNS_TO_DROP, errors="ignore")

In [ ]:
def dataframe_to_parquet(chunk: pd.DataFrame, target_folder_name: str, file_name: str, template_columns: list[str] | None = None):
    chunk = normalize_columns(chunk)
    chunk = drop_unwanted_columns(chunk)

    if template_columns is not None:
        chunk = chunk.reindex(columns=template_columns)

    label = LABEL_COLUMN.lower()
    for col in chunk.columns:
        if col != label:
            chunk[col] = pd.to_numeric(chunk[col], errors="coerce").astype("float64")

    chunk = chunk[chunk[label].str.lower() != label]
    chunk = chunk.dropna(subset=[label])
    output_file = os.path.join(target_folder_name, file_name)
    chunk.to_parquet(output_file, engine="pyarrow", compression="snappy", index=False)

In [ ]:
def create_template_columns(files: list, source_folder_name: str):
    template_columns = []
    seen = set()
    for file_name in files:
        source_file = os.path.join(source_folder_name, file_name)
        cols = pd.read_csv(source_file, nrows=0).columns.str.strip().str.lower().tolist()
        for c in cols:
            if c not in seen and c not in COLUMNS_TO_DROP:
                seen.add(c)
                template_columns.append(c)
    return template_columns

In [ ]:
def get_template_columns(source_folder_name, template_column_path: str = "dataframe-index.pkl"):
    csv_files = get_all_file_names(source_folder_name, "csv")
    template_columns = create_template_columns(csv_files,source_folder_name)
    joblib.dump(template_columns, template_column_path)

In [ ]:
def _convert_single_csv_file(
    source_folder_name: str,
    target_folder_name: str,
    file_name: str,
    template_columns: list[str],
    chunk_size: int,
):
    source_file = os.path.join(source_folder_name, file_name)
    base_name = os.path.splitext(file_name)[0]

    for chunk_number, chunk in enumerate(pd.read_csv(source_file, chunksize=chunk_size, low_memory=False), start=1):
        dataframe_to_parquet(chunk, target_folder_name, f"{base_name}_{chunk_number:05d}.parquet", template_columns)

def convert_all_file_to_parquet(
    source_folder_name: str,
    target_folder_name: str,
    chunk_size: int = CHUNK_SIZE
):
    csv_files = get_all_file_names(source_folder_name, "csv")
    template_columns = create_template_columns(csv_files,source_folder_name)

    total = len(csv_files)
    os.makedirs(target_folder_name, exist_ok=True)

    with parallel_config(backend="loky", inner_max_num_threads=1, verbose=0):
        Parallel(n_jobs=-1, verbose=5)(
            delayed(_convert_single_csv_file)(source_folder_name, target_folder_name, file_name, template_columns, chunk_size)
            for file_name in csv_files
        )

    print(f"Finished processing {total} files.")

In [ ]:
convert_all_file_to_parquet(PATH_FOLDER_CSV, PATH_FOLDER_RAW, CHUNK_SIZE)

## 2.2 Exploratory Data Analysis (EDA)

## 2.3. Change Infinite Values to NaN


Infinite values ($\infty$ and $-\infty$ / `np.inf` and `-np.inf`) are converted to `NaN` values to ensure that they can be handled consistently during the subsequent missing-value imputation process. This transformation allows both originally missing values and invalid infinite values to be processed using the same imputation method.

In [ ]:
def calculate_inf_values(
    source_folder_name: str
):
    parquet_files = get_all_file_names(source_folder_name,"parquet")
    total = len(parquet_files)
    count = 0
    for i, file_name in enumerate(parquet_files, start=1):
        print(f"Processing [{i}/{total}] {file_name}...", end="\r", flush=True)
        source_file = os.path.join(source_folder_name, file_name)
        df = pd.read_parquet(source_file)
        inf_count = np.isinf(df.select_dtypes(include=np.number)).sum().sum()
        count += inf_count
    print("\n")
    print(f"Completed. Processed {total} files.")
    print(f"Found {count} inf values")

In [ ]:
def _change_inf_to_nan_single_file(source_folder_name: str, target_folder_name: str, file_name: str):
    source_file = os.path.join(source_folder_name, file_name)
    target_file = os.path.join(target_folder_name, file_name)

    df = pd.read_parquet(source_file)
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.to_parquet(target_file, index=False)

def change_inf_to_nan(
    source_folder_name: str,
    target_folder_name: str
):
    os.makedirs(target_folder_name, exist_ok=True)

    parquet_files = get_all_file_names(source_folder_name,"parquet")
    total = len(parquet_files)

    with parallel_config(backend="loky", inner_max_num_threads=1, verbose=0):
        Parallel(n_jobs=-1, verbose=5)(
            delayed(_change_inf_to_nan_single_file)(source_folder_name, target_folder_name, file_name)
            for file_name in parquet_files
        )

    print(f"Completed. Processed {total} files.")

In [ ]:
change_inf_to_nan(PATH_FOLDER_RAW, PATH_FOLDER_NO_INF)

### Evaluation

The dataset was cleaned by converting both positive infinite  and negative infinite values ($\infty$ and $-\infty$ / `np.inf` and `-np.inf`) to `NaN`. A total of **121,886 infinite values** were identified across **168 Parquet files** before the cleaning process. After the transformation, no infinite values remained in the dataset.

**Before cleaning:**

In [ ]:
calculate_inf_values(PATH_FOLDER_RAW)

**After cleaning:**

In [ ]:
calculate_inf_values(PATH_FOLDER_NO_INF)

This ensures that all infinite values are handled as missing values and can subsequently be processed during the missing-value imputation stage.


## Train/Validation/Test Split

The CSE-CIC-IDS2018 dataset requires careful consideration when dividing the data into training, validation, and test sets. This is because the attack classes are not uniformly distributed across the dataset; instead, specific attack scenarios were conducted on particular dates. As a result, directly splitting the dataset based on individual days may cause some attack classes to be absent from one or more subsets.

The distribution of attack scenarios across the data collection dates is presented below:

| Date  | Attack(s)                           |
|-------|-------------------------------------|
| 14-02 | FTP-BruteForce, SSH-Bruteforce      |
| 15-02 | DoS-GoldenEye, DoS-Slowloris        |
| 16-02 | DoS-SlowHTTPTest, DoS-Hulk          |
| 20-02 | DDoS-LOIC-HTTP, DDoS-LOIC-UDP       |
| 21-02 | DDoS-LOIC-UDP, DDoS-HOIC            |
| 22-02 | Web Brute Force, XSS, SQL Injection |
| 23-02 | Web Brute Force, XSS, SQL Injection |
| 28-02 | Infiltration                        |
| 01-03 | Infiltration                        |
| 02-03 | Bot                                 |

This distribution indicates that several attack classes are associated with only one or a small number of collection dates. Therefore, assigning entire dates directly to the training, validation, or test set could result in certain attack classes being completely absent from the training data. Such a split would make the experiment evaluate unseen attack-class generalization rather than the intended robustness of the ML-IDS against input disturbances.

Therefore, the primary experiment uses a **stratified train/validation/test split based on the attack label**, ensuring that the attack classes are represented across the three subsets. The validation and test sets are kept separate from the training data to prevent information leakage during model development and final evaluation.

A separate day- or scenario-based split may subsequently be used as an additional experiment to evaluate the model's ability to generalize to traffic collected under different attack scenarios.


In [ ]:
bruteforce = "2018-02-14"
dos_golden = "2018-02-15"
dos_hulk = "2018-02-16"
ddos_http = "2018-02-20"
ddos_udp = "2018-02-21"
web_first = "2018-02-22"
web_second = "2018-02-23"
infiltration_first = "2018-02-28"
infiltration_second = "2018-03-01"
botnet = "2018-03-02"

In [ ]:
def create_train_dev_test_folder(source_folder_name: str, features_folder_name: str, labels_folder_name: str, target_day):

    parquet_files = get_all_file_names(source_folder_name, "parquet")
    parquet_files = filter_file_names(parquet_files, target_day)

    if not parquet_files:
        print(f"No Parquet files found for {target_day}")
        raise ValueError(f"No Parquet files found for {target_day}")
    print(f"Found {len(parquet_files)} files for {target_day}")

    split_names = ("train", "dev", "test")
    feature_folders = {}
    label_folders = {}
    for split_name in split_names:
        feature_folder = os.path.join(features_folder_name, split_name)
        label_folder = os.path.join(labels_folder_name, split_name)
        os.makedirs(feature_folder, exist_ok=True)
        os.makedirs(label_folder, exist_ok=True)
        feature_folders[split_name] = feature_folder
        label_folders[split_name] = label_folder

    return parquet_files, feature_folders, label_folders

In [ ]:
def _split_single_parquet_file(
    source_folder_name: str,
    file_name: str,
    feature_folders: dict[str, str],
    label_folders: dict[str, str],
    label_column: str,
    dev_size: float,
    test_size: float,
    random_state: int,
):
    source_file = os.path.join(source_folder_name, file_name)
    df = pd.read_parquet(source_file)

    train_df, temp_df = train_test_split(
        df,
        test_size=dev_size + test_size,
        random_state=random_state,
        stratify=df[label_column],
    )
    dev_df, test_df = train_test_split(
        temp_df,
        test_size=test_size / (dev_size + test_size),
        random_state=random_state,
        stratify=temp_df[label_column],
    )

    for split_name, split_df in (("train", train_df), ("dev", dev_df), ("test", test_df)):
        labels = split_df[[label_column]]
        features = split_df.drop(columns=[label_column])

        features.to_parquet(os.path.join(feature_folders[split_name], file_name), index=False)
        labels.to_parquet(os.path.join(label_folders[split_name], file_name), index=False)

In [ ]:
def split_parquet_files(
    source_folder_name: str,
    parquet_files: list[str],
    feature_folders: dict[str, str],
    label_folders: dict[str, str],
    label_column: str,
    dev_size: float,
    test_size: float,
    random_state: int,
):
    total = len(parquet_files)

    with parallel_config(backend="loky", inner_max_num_threads=1, verbose=0):
        Parallel(n_jobs=-1, verbose=5)(
            delayed(_split_single_parquet_file)(
                source_folder_name, file_name, feature_folders, label_folders,
                label_column, dev_size, test_size, random_state,
            )
            for file_name in parquet_files
        )

    print(f"Completed splitting {total} files.")

In [ ]:
def split_a_single_day(
    source_folder_name: str,
    features_folder_name: str,
    labels_folder_name: str,
    target_day: str,
    train_size: float = 0.60,
    dev_size: float = 0.20,
    test_size: float = 0.20,
    label_column: str = LABEL_COLUMN.lower(),
    random_state: int = 42,
):
    if min(train_size, dev_size, test_size) < 0.0:
        raise ValueError("train_size, dev_size, test_size must be non-negative")
    if not math.isclose(train_size + dev_size + test_size, 1.0, abs_tol=1e-9):
        raise ValueError("train_size + dev_size + test_size must equal 1.0")

    parquet_files, feature_folders, label_folders = create_train_dev_test_folder(
        source_folder_name, features_folder_name, labels_folder_name, target_day
    )

    split_parquet_files(
        source_folder_name,
        parquet_files,
        feature_folders,
        label_folders,
        label_column,
        dev_size,
        test_size,
        random_state,
    )

In [ ]:
split_a_single_day(PATH_FOLDER_NO_INF, PATH_FOLDER_SPLIT_DATA, PATH_FOLDER_SPLIT_LABEL, bruteforce)

In [ ]:
split_a_single_day(PATH_FOLDER_NO_INF, PATH_FOLDER_SPLIT_DATA, PATH_FOLDER_SPLIT_LABEL, dos_golden)

In [ ]:
split_a_single_day(PATH_FOLDER_NO_INF, PATH_FOLDER_SPLIT_DATA, PATH_FOLDER_SPLIT_LABEL, dos_hulk)

In [ ]:
split_a_single_day(PATH_FOLDER_NO_INF, PATH_FOLDER_SPLIT_DATA, PATH_FOLDER_SPLIT_LABEL, ddos_http)

In [ ]:
split_a_single_day(PATH_FOLDER_NO_INF, PATH_FOLDER_SPLIT_DATA, PATH_FOLDER_SPLIT_LABEL, ddos_udp)

In [ ]:
split_a_single_day(PATH_FOLDER_NO_INF, PATH_FOLDER_SPLIT_DATA, PATH_FOLDER_SPLIT_LABEL, web_first)

In [ ]:
split_a_single_day(PATH_FOLDER_NO_INF, PATH_FOLDER_SPLIT_DATA, PATH_FOLDER_SPLIT_LABEL, web_second)

In [ ]:
split_a_single_day(PATH_FOLDER_NO_INF, PATH_FOLDER_SPLIT_DATA, PATH_FOLDER_SPLIT_LABEL, infiltration_first)

In [ ]:
split_a_single_day(PATH_FOLDER_NO_INF, PATH_FOLDER_SPLIT_DATA, PATH_FOLDER_SPLIT_LABEL, infiltration_second)

In [ ]:
split_a_single_day(PATH_FOLDER_NO_INF, PATH_FOLDER_SPLIT_DATA, PATH_FOLDER_SPLIT_LABEL, botnet)

Files are successfully split.

## Imputation

Missing values are filled using **per-column medians computed from the training split only**, then applied to train/dev/test. Using the median (rather than the mean) avoids distortion from the heavy outliers typical of network flow features (e.g. flow duration, byte counts). Fitting the medians on the training split only and reusing them for dev/test prevents information leakage from the validation/test sets into preprocessing.

### Create Imputation

imputation for NaN

In [ ]:
def get_numeric_columns(combined: pl.DataFrame) -> list[str]:
    schema = combined.collect_schema()

    numeric_cols = []
    for column, dtype in zip(schema.names(), schema.dtypes()):
        if dtype.is_numeric():
            numeric_cols.append(column)

    print(f"Found {len(numeric_cols)} numeric columns")
    return numeric_cols

In [ ]:
def compute_medians(combined: pl.DataFrame, numeric_cols: list[str]) -> pl.DataFrame:
    median_exprs = []
    total = len(numeric_cols)

    for i,column in enumerate(numeric_cols,start=1):
        print(f"Processing [{i}/{total}] {column}...", end="\r", flush=True)
        expr = pl.col(column).median().alias(column)
        median_exprs.append(expr)

    medians = combined.select(median_exprs)
    return medians

In [ ]:
def build_median_dict(medians: pl.DataFrame, numeric_cols: list[str]) -> dict[str, float]:
    median_dict = {}
    null_columns = []

    for c in numeric_cols:
        val = medians[c][0]
        if val is None:
            null_columns.append(c)
            median_dict[c] = 0.0
        else:
            median_dict[c] = float(val)

    if null_columns:
        print(f"Warning: {len(null_columns)} columns had no non-null values, defaulted to 0.0: {null_columns}")

    return median_dict

In [ ]:
def save_median_dict(median_dict: dict[str, float], output_file: str):
    with open(output_file, "w") as file:
        json.dump(median_dict, file, indent=4)
    print(f"Saved medians to: {output_file}")

In [ ]:
def get_median_imputation(source_folder_name: str, output_file: str) -> dict[str, float]:
    df = get_polars_data_frame_without_label(source_folder_name,"train")
    numeric_cols = get_numeric_columns(df)
    medians = compute_medians(df, numeric_cols)
    median_dict = build_median_dict(medians, numeric_cols)
    save_median_dict(median_dict, output_file)
    return median_dict

In [ ]:
median = get_median_imputation(PATH_FOLDER_SPLIT_DATA, PATH_IMPUTER)
print(json.dumps(median, indent=4))

### Impute Training Data

In [ ]:
PATH_FOLDER_IMPUTED = "data-imputed"

In [ ]:
def impute_dataframe(df:pd.DataFrame, medians):
    for column, median in medians.items():
        if column in df.columns:
            df[column] = df[column].fillna(median)
    return df

In [ ]:
def _impute_single_file(source_file: str, output_split_folder: str, medians: dict[str, float]):
    df = pd.read_parquet(source_file)
    df = impute_dataframe(df, medians)
    output_file = os.path.join(output_split_folder, os.path.basename(source_file))
    df.to_parquet(
        output_file,
        engine="pyarrow",
        compression="snappy",
        index=False
    )

def impute_all_split_files(source_folder_name: str, output_folder_name: str, medians_path: str):
    with open(medians_path, "r") as f:
        medians = json.load(f)

    for split_name in ("train", "dev", "test"):
        files = get_split_parquet_files(source_folder_name, split_name)
        total = len(files)
        output_split_folder = os.path.join(output_folder_name, split_name)
        os.makedirs(output_split_folder, exist_ok=True)

        with parallel_config(backend="loky", inner_max_num_threads=1, verbose=0):
            Parallel(n_jobs=-1, verbose=5)(
                delayed(_impute_single_file)(file, output_split_folder, medians)
                for file in files
            )

        print(f"Finished imputing {total} {split_name} files.")

In [ ]:
impute_all_split_files(PATH_FOLDER_SPLIT_DATA, PATH_FOLDER_IMPUTED, PATH_IMPUTER)

## Normalized or Feature Scaling

Features are standardized (zero mean, unit variance) using `StandardScaler`. As with imputation, the scaler is fit with `partial_fit` on the **train split only** (processed file-by-file to avoid loading the full ~16M-row dataset into memory at once), then reused to transform train/dev/test. This keeps dev/test statistically unseen during fitting and puts every feature on a comparable scale, which PCA and distance/gradient-based models both depend on.

### Create Scaler

In [ ]:
def fit_scaler_on_files(scaler: StandardScaler, file_paths: list[str]) -> StandardScaler:
    total = len(file_paths)
    for i, file_path in enumerate(file_paths, start=1):
        print(f"Processing [{i}/{total}] {file_path}...", end="\r", flush=True)
        df = pd.read_parquet(file_path)

        if df.empty:
            print(f"Empty file found: {file_path}")
            continue

        scaler.partial_fit(df)
    return scaler

In [ ]:
def create_standard_scaler(source_folder_name: str, split_name: str = "train") -> StandardScaler:
    file_paths = get_split_parquet_files(source_folder_name, split_name)
    scaler = StandardScaler()
    scaler = fit_scaler_on_files(scaler, file_paths)

    if not hasattr(scaler, "mean_"):
        raise ValueError("The scaler could not be fitted because all files were empty.")

    print(" " * 100, end="\r")
    print(f"Successfully fitted scaler using {len(file_paths)} Parquet file(s) from '{split_name}'.")

    return scaler

In [ ]:
def dump_standard_scaler(scaler: StandardScaler,output_path:str):
    joblib.dump(scaler,output_path)

In [ ]:
def get_standard_scaler(source_folder_name:str,output_path:str):
    scaler = create_standard_scaler(source_folder_name)
    dump_standard_scaler(scaler,output_path)

In [ ]:
get_standard_scaler(PATH_FOLDER_IMPUTED, PATH_SCALER)

### Scale Training Data

In [ ]:
def load_standard_scaler(path:str)->StandardScaler:
    return joblib.load(path)

In [ ]:
def _scale_single_file(file_path: str, scaler: StandardScaler, output_split_folder: str):
    df = pd.read_parquet(file_path)
    columns = df.columns.tolist()

    data = scaler.transform(df)
    scaled_df = pd.DataFrame(data, columns=columns)

    scaled_df.to_parquet(
        os.path.join(output_split_folder, os.path.basename(file_path)),
        engine="pyarrow",
        compression="snappy",
        index=False
    )

def scale_split_files(source_folder_name: str, scaler_path: str, output_folder_name: str):
    scaler = load_standard_scaler(scaler_path)

    for split_name in ("train", "dev", "test"):
        file_paths = get_split_parquet_files(source_folder_name, split_name)
        output_split_folder = os.path.join(output_folder_name, split_name)
        os.makedirs(output_split_folder, exist_ok=True)
        total = len(file_paths)

        with parallel_config(backend="loky", inner_max_num_threads=1):
            Parallel(n_jobs=-1, verbose=5)(
                delayed(_scale_single_file)(file_path, scaler, output_split_folder)
                for file_path in file_paths
            )

        print(f"Finished scaling {total} {split_name} files.")

In [ ]:
scale_split_files(PATH_FOLDER_IMPUTED, PATH_SCALER, PATH_FOLDER_SCALED)

## Principal Component Analysis (PCA)

`IncrementalPCA` is used instead of standard `PCA` because it supports `partial_fit` on mini-batches, so the scaled dataset never needs to be fully loaded into memory. Transformers are fit for a range of component counts (5-40) so the explained-variance-vs-components trade-off can be inspected before committing to a final value.

### Creating Incremental Principal Component Analysis

In [ ]:
LIST_PC_COMPONENTS = [5,10,15,20,25,30,35,40]

In [ ]:
def fit_ipca_on_files(ipca: IncrementalPCA, file_paths: list[str]) -> IncrementalPCA:
    total = len(file_paths)
    for i, file_path in enumerate(file_paths, start=1):
        print(f"Processing [{i}/{total}] {file_path}...", end="\r", flush=True)
        df = pd.read_parquet(file_path)

        if df.empty:
            print(f"Empty file found: {file_path}")
            continue

        ipca.partial_fit(df)
    print(" " * 100, end="\r")
    return ipca

In [ ]:
def create_ipca(n_components,source_folder_name: str, split_name: str = "train") -> IncrementalPCA:
    print(f"Creating IPCA transfromer with {n_components} pc")
    file_paths = get_split_parquet_files(source_folder_name, split_name)
    ipca = IncrementalPCA(n_components=n_components)
    ipca = fit_ipca_on_files(ipca, file_paths)
    print(f"Successfully fitted ipca with {n_components} pc from '{split_name}'.")

    return ipca

In [ ]:
def get_ipca_transformer_path(pc:int, ipca_path:str)->str:
    return f"{pc}-pc-{ipca_path}"

In [ ]:
def dump_ipca(ipca: IncrementalPCA, transformer_folder:str = PATH_FOLDER_IPCA_TRANSFORMER,ipca_path:str = PATH_IPCA):
    filename = get_ipca_transformer_path(ipca.n_components_,ipca_path)
    joblib.dump(ipca,os.path.join(transformer_folder, filename))

In [ ]:
def get_ipca(list_pc_components: list, source_folder_name: str, output_path: str):
    os.makedirs(PATH_FOLDER_IPCA_TRANSFORMER, exist_ok=True)
    for pc in list_pc_components:
        ipca = create_ipca(pc, source_folder_name)
        dump_ipca(ipca, PATH_FOLDER_IPCA_TRANSFORMER, output_path)

In [ ]:
get_ipca(LIST_PC_COMPONENTS,PATH_FOLDER_SCALED, PATH_IPCA)

### Evaluate Incremental Principal Component Analysis

In [ ]:
def load_ipca(pc:int,transformer_folder:str = PATH_FOLDER_IPCA_TRANSFORMER,ipca_path:str = PATH_IPCA) -> IncrementalPCA:
    filename = get_ipca_transformer_path(pc,ipca_path)
    return joblib.load(os.path.join(transformer_folder, filename))

In [ ]:
def evaluate_ipca_variance(list_pc_components: list) -> pd.DataFrame:
    rows = []
    for pc in list_pc_components:
        ipca = load_ipca(pc)

        cum_var = np.cumsum(ipca.explained_variance_ratio_)
        rows.append({
            "n_components": pc,
            "total_explained_variance": cum_var[-1],
            "explained_variance_ratio": ipca.explained_variance_ratio_,
        })

    return pd.DataFrame(rows)

In [ ]:
def plot_ipca_variance(evaluation_results: pd.DataFrame):
    plt.figure(figsize=(8, 5))

    x = evaluation_results["n_components"]
    y = evaluation_results["total_explained_variance"]
    plt.plot(x, y, marker="o", color="black")
    for xi, yi in zip(x, y):
        plt.annotate(f"{yi:.3f}", (xi, yi), xytext=(0, 8), textcoords="offset points", ha="center")

    plt.axhline(0.95, color="red", linestyle="--", label="95% threshold")
    plt.xlabel("Principal Components")
    plt.ylabel("Cumulative Explained Variance")
    plt.title("IPCA: Variance Retained vs. Principal Components")
    plt.legend()
    plt.grid(True)

    plt.show()

In [ ]:
ipca_variance_results = evaluate_ipca_variance(LIST_PC_COMPONENTS)
plot_ipca_variance(ipca_variance_results)

### Implement Incremental Principal Component Analysis

`IPCA_PC_USED = 25` was chosen from the variance plot above as the smallest evaluated component count that clears the 95% cumulative explained-variance threshold.

In [ ]:
IPCA_PC_USED = 25

In [ ]:
def _ipca_transform_single_file(file_path: str, ipca: IncrementalPCA, output_split_folder: str):
    df = pd.read_parquet(file_path)

    data = ipca.transform(df)
    pc_columns = [f"pc{i+1}" for i in range(data.shape[1])]
    scaled_df = pd.DataFrame(data, columns=pc_columns)

    scaled_df.to_parquet(
        os.path.join(output_split_folder, os.path.basename(file_path)),
        engine="pyarrow",
        compression="snappy",
        index=False
    )

def ipca_split_files(source_folder_name: str, ipca_path: str, output_folder_name: str):
    ipca = load_ipca(IPCA_PC_USED)

    for split_name in ("train", "dev", "test"):
        file_paths = get_split_parquet_files(source_folder_name, split_name)
        output_split_folder = os.path.join(output_folder_name, split_name)
        os.makedirs(output_split_folder, exist_ok=True)
        total = len(file_paths)

        with parallel_config(backend="loky", inner_max_num_threads=1):
            Parallel(n_jobs=-1, verbose=5)(
                delayed(_ipca_transform_single_file)(file_path, ipca, output_split_folder)
                for file_path in file_paths
            )

        print(f"Finished scaling {total} {split_name} files.")

In [ ]:
ipca_split_files(PATH_FOLDER_SCALED, PATH_IPCA, PATH_FOLDER_IPCA)

## Synthetic Minor Oversampling Technique

CSE-CIC-IDS2018 is heavily imbalanced (Benign traffic dominates; some attack classes are a small fraction of a percent). SMOTE is applied to the **training split only** (never dev/test, so evaluation still reflects real-world class balance) to synthesize minority-class samples and reduce the model's bias toward the majority class.

SMOTE resamples the **PCA-reduced train features** (`PATH_FOLDER_IPCA`, paired with the original `PATH_FOLDER_SPLIT_LABEL` labels), since that's the final feature representation a downstream model would train on. Fitting concatenates every train file via polars (`load_train_features_and_labels`), since imbalanced-learn needs the full split in memory to compute global class counts. Once fitted, the transformer is applied **one file at a time** (`load_smote`, `resample_with_smote`, `save_smote_train_data`) using pandas instead - matching the streaming approach used for IPCA/normalization elsewhere - so resampling never needs the whole split in memory at once; imbalanced-learn only accepts numpy arrays, so the conversion happens right at the `fit_resample` call and the result is wrapped back into a `pd.DataFrame`/`pd.Series`.

Because some attack classes (e.g. SQL Injection) have very few samples in this dataset, `fit_smote` clamps `k_neighbors` down to what the smallest class can support instead of letting `fit_resample` raise on the default `k_neighbors=5`. Fitting is kept separate from resampling: `fit_smote` fits and `dump_smote` persists the fitted transformer via joblib right away, and `resample_with_smote` performs the actual oversampling per file afterwards. Two things can still go wrong at the per-file level that the global fit can't see: (1) a class that's fine globally can be scarce in one specific chunk (e.g. one file has only 5 `Benign` rows against 59,995 `DDOS attack-HOIC` rows), so `resample_with_smote` re-checks `k_neighbors` against that file's own smallest class and clamps further if needed; (2) since each day's capture is chunked in original time order, most individual chunks fall entirely outside the attack window and end up **100% Benign** (123 of the 168 train files, in practice) - `fit_resample` requires at least 2 classes, so `resample_with_smote` detects a single-class file and passes it through unresampled rather than erroring.

### Create SMOTE

In [ ]:
def load_train_features_and_labels(features_folder_name: str, labels_folder_name: str, split_name: str = "train") -> tuple[pl.DataFrame, pl.Series]:
    feature_files = get_split_parquet_files(features_folder_name, split_name)
    label_files = get_split_parquet_files(labels_folder_name, split_name)

    print(f"Loading {len(feature_files)} '{split_name}' feature file(s)...", end="", flush=True)
    features_df = pl.concat([pl.scan_parquet(f) for f in feature_files]).collect()
    labels_df = pl.concat([pl.scan_parquet(f) for f in label_files]).select(LABEL_COLUMN.lower()).collect()

    print(f"Loaded {features_df.height} rows for '{split_name}'.")
    return features_df, labels_df.to_series()

In [ ]:
def get_class_count(labels: pl.Series) -> dict:
    counts = labels.value_counts()
    return dict(zip(counts[labels.name].to_list(), counts["count"].to_list()))

In [ ]:
def fit_smote(features: pl.DataFrame, labels: pl.Series, random_state: int = 42) -> SMOTE:
    class_counts = get_class_count(labels)
    smallest_class_count = min(class_counts.values())
    k_neighbors = max(1, min(5, smallest_class_count - 1))
    
    if k_neighbors < 5:
        print(f"Warning: smallest class has {smallest_class_count} samples; reducing k_neighbors to {k_neighbors}")

    smote = SMOTE(random_state=random_state, k_neighbors=k_neighbors)
    with parallel_config(n_jobs=-1):
        smote.fit(features.to_numpy(), labels.to_numpy())
    return smote

In [ ]:
def dump_smote(smote: SMOTE, transformer_folder: str = PATH_FOLDER_SMOTE_TRANSFORMER, smote_path: str = PATH_SMOTE):
    os.makedirs(transformer_folder, exist_ok=True)
    output_path = os.path.join(transformer_folder, smote_path)
    joblib.dump(smote, output_path)
    print(f"Saved fitted SMOTE transformer to: {output_path}")

In [ ]:
def get_smote_transformer(features_folder_name: str, labels_folder_name: str):
    features, labels = load_train_features_and_labels(features_folder_name, labels_folder_name)
    smote = fit_smote(features, labels)
    dump_smote(smote)

In [ ]:
get_smote_transformer(PATH_FOLDER_IPCA, PATH_FOLDER_SPLIT_LABEL)

### Implement SMOTE

In [ ]:
def load_smote(transformer_folder: str = PATH_FOLDER_SMOTE_TRANSFORMER, smote_path: str = PATH_SMOTE) -> SMOTE:
    return joblib.load(os.path.join(transformer_folder, smote_path))

In [ ]:
def resample_with_smote(smote: SMOTE, features: pd.DataFrame, labels: pd.Series) -> tuple[pd.DataFrame, pd.Series]:
    if labels.nunique() < 2:
        return features, labels

    smallest_class_count = labels.value_counts().min()
    k_neighbors = max(1, min(smote.k_neighbors, smallest_class_count - 1))

    if k_neighbors < smote.k_neighbors:
        print(f"Warning: smallest class in this file has {smallest_class_count} samples; reducing k_neighbors to {k_neighbors}")
        smote = SMOTE(random_state=smote.random_state, k_neighbors=k_neighbors, sampling_strategy=smote.sampling_strategy)

    features_resampled, labels_resampled = cast(
        tuple[np.ndarray, np.ndarray],
        smote.fit_resample(features.to_numpy(), labels.to_numpy()),
    )
    return pd.DataFrame(features_resampled, columns=features.columns), pd.Series(labels_resampled, name=labels.name)

In [ ]:
def save_smote_train_data(features: pd.DataFrame, labels: pd.Series, file_name: str, output_folder_name: str = PATH_FOLDER_SMOTE):
    output_split_folder = os.path.join(output_folder_name, "train")
    os.makedirs(output_split_folder, exist_ok=True)

    output_df = features.assign(**{LABEL_COLUMN.lower(): labels.to_numpy()})
    output_df.to_parquet(
        os.path.join(output_split_folder, file_name),
        engine="pyarrow",
        compression="snappy",
        index=False
    )

In [ ]:
def _smote_resample_single_file(feature_file_path: str, label_file_path: str, smote: SMOTE, output_folder_name: str):
    features = pd.read_parquet(feature_file_path)
    labels = pd.read_parquet(label_file_path)[LABEL_COLUMN.lower()]

    features_resampled, labels_resampled = resample_with_smote(smote, features, labels)
    save_smote_train_data(features_resampled, labels_resampled, os.path.basename(feature_file_path), output_folder_name)

def smote_resample_files(features_folder_name: str, labels_folder_name: str, output_folder_name: str = PATH_FOLDER_SMOTE):
    smote = load_smote()

    feature_files = get_split_parquet_files(features_folder_name, "train")
    label_files = get_split_parquet_files(labels_folder_name, "train")
    total = len(feature_files)

    with parallel_config(backend="loky", inner_max_num_threads=1, verbose=0):
        Parallel(n_jobs=-1, verbose=5)(
            delayed(_smote_resample_single_file)(feature_file_path, label_file_path, smote, output_folder_name)
            for feature_file_path, label_file_path in zip(feature_files, label_files)
        )

    print(f"Finished SMOTE-resampling {total} train file(s).")

In [ ]:
smote_resample_files(PATH_FOLDER_IPCA, PATH_FOLDER_SPLIT_LABEL, PATH_FOLDER_SMOTE)

### Evaluate SMOTE

In [ ]:
def get_class_count_from_label(source_folder_path:str): 
    label_files = get_split_parquet_files(source_folder_path, "train")
    labels = pl.concat([pl.scan_parquet(f) for f in label_files]).select(LABEL_COLUMN.lower()).collect()
    return get_class_count(labels.to_series()), len(labels)

In [ ]:
before_class_count,before_row_count = get_class_count_from_label(PATH_FOLDER_SPLIT_LABEL)
print(f"Row count: {before_row_count}")
print(f"Class count:\n{json.dumps(before_class_count, indent=4)}")

In [ ]:
after_class_count,after_row_count = get_class_count_from_label(PATH_FOLDER_SMOTE)
print(f"Row count: {after_row_count}")
print(f"Class count:\n{json.dumps(after_class_count, indent=4)}")

In [ ]:
def plot_smote_comparison(before_class_count: dict[str, int], after_class_count: dict[str, int],) -> None:
    classes = sorted(set(before_class_count) | set(after_class_count))
    before = []
    for class_name in classes:
        before.append(before_class_count.get(class_name, 0))

    after = []
    for class_name in classes:
        after.append(after_class_count.get(class_name, 0))

    df = pd.DataFrame({"Class": classes, "Before SMOTE": before, "After SMOTE": after,})

    ax = df.set_index("Class").plot(
        kind="bar",
        figsize=(14, 7),
        width=0.8,
    )

    ax.set_title("Class Distribution Before and After SMOTE")
    ax.set_xlabel("Class")
    ax.set_ylabel("Number of Samples")

    plt.xticks(rotation=45, ha="right")
    plt.legend(title="Dataset")
    plt.tight_layout()
    plt.show()

In [ ]:
plot_smote_comparison(before_class_count,after_class_count)

# 3. Train Model

#### Load Train Data

In [30]:
train_x, train_y = get_polars_data_frame_with_label(PATH_FOLDER_SMOTE)

Found 168 train files


#### Load Val/Dev Data

In [31]:
dev_x = get_polars_data_frame_without_label(PATH_FOLDER_IPCA,"dev")
dev_y = get_polars_data_frame_without_label(PATH_FOLDER_SPLIT_LABEL,"dev").to_series()

Found 168 dev files
Found 168 dev files] data-ipca\dev\2018-03-02-Friday_TrafficForML_CICFlowMeter_00011.parquet......


## Load Test Data

In [33]:
test_x = get_polars_data_frame_without_label(PATH_FOLDER_IPCA,"test")
test_y = get_polars_data_frame_without_label(PATH_FOLDER_SPLIT_LABEL,"test").to_series()

Found 168 test files
Found 168 test files data-ipca\test\2018-03-02-Friday_TrafficForML_CICFlowMeter_00011.parquet......


## Create Sub Sample

In [34]:
def get_stratified_subsample(
        feature: pl.DataFrame,
        label: pl.Series,
        subsample_size: int | None,
        random_state: int,
    ) -> tuple[pl.DataFrame, pl.Series]:
    """Draw a random subsample whose per-class proportions match the input.

    Each class keeps at least one row, so rare attack classes never drop out
    of the subsample. The returned frame and series stay row-aligned, and the
    rows are shuffled so classes are not left in contiguous blocks.
    """
    total_rows = feature.height

    if subsample_size is None or subsample_size >= total_rows:
        return feature, label

    label_column = "__stratify_label__"
    working = feature.with_columns(label.alias(label_column)) # Adding label column to feature dataframe

    class_values = label.unique(maintain_order=True).to_list()
    sampled_parts = []

    for class_value in class_values:
        class_rows = working.filter(pl.col(label_column) == class_value)

        target_count = round(class_rows.height * subsample_size / total_rows)
        target_count = max(1, target_count)
        target_count = min(target_count, class_rows.height)

        class_sample = class_rows.sample(n=target_count, seed=random_state, shuffle=True)
        sampled_parts.append(class_sample)

    subsample = pl.concat(sampled_parts)
    subsample = subsample.sample(fraction=1.0, seed=random_state, shuffle=True)

    sampled_label = subsample.get_column(label_column).rename(label.name) # Extacting label
    sampled_feature = subsample.drop(label_column) # Dropping label
    return sampled_feature, sampled_label

## Create Evaluation Report

In [35]:
def evaluate_results(true_label, prediction_label, labels=None):
    if labels is None:
        labels = np.unique(np.concatenate([true_label, prediction_label]))

    label_list: list[Any] = list(labels)

    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        true_label, prediction_label, labels=label_list,
        average="macro", zero_division=0
    )
    w_p, w_r, w_f1, _ = precision_recall_fscore_support(
        true_label, prediction_label, labels=label_list,
        average="weighted", zero_division=0
    )
    per_p, per_r, per_f1, support = precision_recall_fscore_support(
        true_label, prediction_label, labels=label_list,
        average=None, zero_division=0
    )

    per_class_precision: dict[Any, float] = {
        lbl: float(v) for lbl, v in zip(label_list, per_p)
    }
    per_class_recall: dict[Any, float] = {
        lbl: float(v) for lbl, v in zip(label_list, per_r)
    }
    per_class_f1: dict[Any, float] = {
        lbl: float(v) for lbl, v in zip(label_list, per_f1)
    }
    per_class_support: dict[Any, int] = {
        lbl: int(v) for lbl, v in zip(label_list, support)
    }

    return {
        "labels": label_list,
        "accuracy": float(accuracy_score(true_label, prediction_label)),
        "macro_precision": float(macro_p),
        "macro_recall": float(macro_r),
        "macro_f1": float(macro_f1),
        "weighted_precision": float(w_p),
        "weighted_recall": float(w_r),
        "weighted_f1": float(w_f1),
        "per_class_precision": per_class_precision,
        "per_class_recall": per_class_recall,
        "per_class_f1": per_class_f1,
        "per_class_support": per_class_support,
        "confusion_matrix": confusion_matrix(
            true_label, prediction_label, labels=label_list
        ),
    }

In [36]:
def per_class_report(report: dict) -> pd.DataFrame:
    """Turn an `evaluate_results` dict into a readable per-class table.

    Prints the headline accuracy / macro / weighted averages, then returns the
    per-class precision/recall/F1/support sorted worst-F1 first -- so the weak
    (usually rare attack) classes sit at the top instead of being averaged away
    by the dominant Benign class.
    """
    print(f"accuracy           = {report['accuracy']:.4f}")
    print(f"macro    P / R / F1 = {report['macro_precision']:.4f} / {report['macro_recall']:.4f} / {report['macro_f1']:.4f}")
    print(f"weighted P / R / F1 = {report['weighted_precision']:.4f} / {report['weighted_recall']:.4f} / {report['weighted_f1']:.4f}")

    per_class = pd.DataFrame({
        "class": list(report["per_class_f1"].keys()),
        "precision": list(report["per_class_precision"].values()),
        "recall": list(report["per_class_recall"].values()),
        "f1": list(report["per_class_f1"].values()),
        "support": list(report["per_class_support"].values()),
    })
    return per_class.sort_values("f1", ascending=True, ignore_index=True)

In [37]:
def plot_confusion_matrix(report: dict, normalize: bool = True, figsize: tuple[int, int] = (11, 9)) -> None:
    """Heatmap of the confusion matrix carried in an `evaluate_results` dict.

    Row-normalised by default: each row then shows the share of that true
    class's samples that landed in each predicted class. Without it the Benign
    row saturates the colour scale and every attack-class mistake reads as zero.
    """
    matrix = np.asarray(report["confusion_matrix"], dtype=float)
    labels = [str(label) for label in report["labels"]]

    if normalize:
        row_sums = matrix.sum(axis=1, keepdims=True)
        matrix = np.divide(matrix, row_sums, out=np.zeros_like(matrix), where=row_sums != 0)

    fig, ax = plt.subplots(figsize=figsize)
    image = ax.imshow(matrix, cmap="Blues", vmin=0, vmax=1 if normalize else None)

    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_yticklabels(labels)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_title("Confusion matrix" + (" (row-normalised)" if normalize else ""))

    text_threshold = matrix.max() / 2.0
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            value = matrix[i, j]
            cell_text = f"{value:.2f}" if normalize else f"{int(value)}"
            ax.text(j, i, cell_text, ha="center", va="center",
                    color="white" if value > text_threshold else "black", fontsize=8)

    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()

## 3.1. K-Nearest Neighbor

In [38]:
list_k_neighbors = list(range(1, 16, 2)) 
list_weight = ["uniform", "distance"]
list_metric = ["euclidean", "manhattan"]

In [39]:
def get_knn_name(k_neighbors:int,weight,metric)-> str:
    return f"KNN-k-{k_neighbors}-w-{weight}-m-{metric}.pkl"

### Hyperparameter Search

In [40]:
KNN_TRAIN_SUBSAMPLE = 1_000_000
KNN_DEV_SUBSAMPLE = 200_000
KNN_QUERY_CHUNK = 100_000
KNN_RANDOM_STATE = 42

#### Helper functions for Nearest Neighbor

In [36]:
def _distance_to_weight(distances: np.ndarray) -> np.ndarray:
    """Inverse-distance vote weights, matching sklearn's zero-distance rule.

    If a dev row sits exactly on one or more train points, only those exact
    matches vote (weight 1) and every other neighbour gets weight 0.
    """
    with np.errstate(divide="ignore"):
        weights = 1.0 / distances

    exact_match_mask = np.isinf(weights)
    row_has_exact_match = np.any(exact_match_mask, axis=1)

    weights[row_has_exact_match] = exact_match_mask[row_has_exact_match]
    return weights

In [37]:
def _weighted_majority_vote(neighbor_labels: np.ndarray, vote_weights: np.ndarray) -> np.ndarray:
    """For each row, return the label class with the largest summed weight."""
    class_values = np.unique(neighbor_labels)
    row_count = neighbor_labels.shape[0]

    score_per_class = np.zeros((row_count, class_values.shape[0]))

    for column, class_value in enumerate(class_values):
        belongs_to_class = neighbor_labels == class_value
        score_per_class[:, column] = np.sum(belongs_to_class * vote_weights, axis=1)

    winning_column = np.argmax(score_per_class, axis=1)
    return class_values[winning_column]

In [38]:
def get_predictions(
        neighbor_indices: np.ndarray,
        neighbor_distances: np.ndarray,
        train_label_encoded: np.ndarray,
        k_neighbors: int,
        weight: str,
    ) -> np.ndarray:
    """Vote one predicted label per dev row from a precomputed neighbour query.

    `neighbor_indices` / `neighbor_distances` come from a NearestNeighbors index
    fitted at the largest k in the grid. Only the first `k_neighbors` columns are
    read here, so a single query is reused for every smaller k.
    """
    nearest_indices = neighbor_indices[:, :k_neighbors]
    nearest_distances = neighbor_distances[:, :k_neighbors]

    nearest_labels = train_label_encoded[nearest_indices]

    if weight == "uniform":
        vote_weights = np.ones_like(nearest_distances)
    else:
        vote_weights = _distance_to_weight(nearest_distances)

    return _weighted_majority_vote(nearest_labels, vote_weights)

In [39]:
def get_nearest_neighbors(
        train_feature: np.ndarray,
        metric: str,
        max_k: int,
    ) -> NearestNeighbors:
    """Fit one brute-force neighbour index for a single distance metric.

    Fitted at `max_k` (the largest k in the grid) so one query can be sliced
    down for every smaller k. The brute-force search is threaded by OpenMP and
    ignores `n_jobs` once dispatched (see check_openmp_threads).
    """
    index = NearestNeighbors(
        n_neighbors=max_k,
        metric=metric,
        algorithm="brute",
        n_jobs=-1,
    )
    index.fit(train_feature)
    return index

In [40]:
def _encode_labels(label_values: np.ndarray, class_values: np.ndarray) -> np.ndarray:
    """Map label strings to the integer positions used in `class_values`.

    Labels missing from `class_values` (unseen in the train subsample) map to
    -1 and therefore never match a prediction.
    """
    class_to_code = {}
    for code, class_value in enumerate(class_values):
        class_to_code[class_value] = code

    codes = []
    for label_value in label_values:
        codes.append(class_to_code.get(label_value, -1))

    return np.asarray(codes)

In [41]:
def _query_neighbors_in_chunks(
        index: NearestNeighbors,
        query_feature: np.ndarray,
        chunk_size: int,
    ) -> tuple[np.ndarray, np.ndarray]:
    """Run `index.kneighbors` over the query rows in chunks to cap peak memory."""
    distance_chunks = []
    index_chunks = []

    total_rows = query_feature.shape[0]

    for start in range(0, total_rows, chunk_size):
        stop = min(start + chunk_size, total_rows)
        chunk_distances, chunk_indices = index.kneighbors(query_feature[start:stop])
        distance_chunks.append(chunk_distances)
        index_chunks.append(chunk_indices)

    neighbor_distances = np.vstack(distance_chunks)
    neighbor_indices = np.vstack(index_chunks)
    return neighbor_distances, neighbor_indices

In [42]:
def _score_hyperparameter_combo(
        neighbor_indices: np.ndarray,
        neighbor_distances: np.ndarray,
        train_label_encoded: np.ndarray,
        dev_label_encoded: np.ndarray,
        k_neighbors: int,
        weight: str,
        metric: str,
        n_classes: int,
    ) -> dict:
    """Vote predictions for one (k, weight, metric) point and score them on dev.

    Reports both the macro average (every class weighted equally -- the honest
    metric for a heavily imbalanced IDS) and the weighted average (dominated by
    the majority Benign class), plus the full per-class precision/recall/F1/
    support vectors indexed by encoded class code. `labels=range(n_classes)`
    keeps every vector the same length and aligned across combos even when a
    combo never predicts (or never sees) some class.
    """
    predictions = get_predictions(
        neighbor_indices,
        neighbor_distances,
        train_label_encoded,
        k_neighbors,
        weight,
    )

    labels = np.arange(n_classes)

    accuracy = accuracy_score(dev_label_encoded, predictions)
    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        dev_label_encoded, predictions, labels=labels, average="macro", zero_division=0,
    )
    weighted_precision, weighted_recall, weighted_f1, _ = precision_recall_fscore_support(
        dev_label_encoded, predictions, labels=labels, average="weighted", zero_division=0,
    )
    per_class_precision, per_class_recall, per_class_f1, per_class_support = (
        precision_recall_fscore_support(
            dev_label_encoded, predictions, labels=labels, average=None, zero_division=0,
        )
    )

    return {
        "k_neighbors": k_neighbors,
        "weight": weight,
        "metric": metric,
        "accuracy": accuracy,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
        "weighted_precision": weighted_precision,
        "weighted_recall": weighted_recall,
        "weighted_f1": weighted_f1,
        "per_class_precision": per_class_precision,
        "per_class_recall": per_class_recall,
        "per_class_f1": per_class_f1,
        "per_class_support": per_class_support,
    }

#### Searching

In [43]:
def knn_hyperparameter_grid_search(
        train_feature: pl.DataFrame,
        train_label: pl.Series,
        dev_feature: pl.DataFrame,
        dev_label: pl.Series,
        list_k: list[int],
        weights: list[str] | None = None,
        metrics: list[str] | None = None,
        train_subsample: int | None = KNN_TRAIN_SUBSAMPLE,
        dev_subsample: int | None = KNN_DEV_SUBSAMPLE,
        query_chunk: int = KNN_QUERY_CHUNK,
        random_state: int = KNN_RANDOM_STATE,
        n_jobs: int = -1,
    ) -> pd.DataFrame:
    """Score every (k, weight, metric) combination on a stratified dev subsample.

    Train and dev are each stratified-subsampled first. For every metric a single
    NearestNeighbors index is fitted at max(list_k) and queried once; the cheap
    vote-and-score step for each (k, weight) pair then runs across threads,
    sharing that one fitted index.

    Rows are returned sorted by macro F1. Each row also carries the weighted
    averages and the per-class precision/recall/F1/support vectors; the dev-split
    class names (aligned with those vectors) are stored on
    result.attrs["class_names"] for knn_per_class_report.
    """
    if weights is None:
        weights = list_weight
    if metrics is None:
        metrics = list_metric

    train_feature_sub, train_label_sub = get_stratified_subsample(
        train_feature, train_label, train_subsample, random_state
    )
    dev_feature_sub, dev_label_sub = get_stratified_subsample(
        dev_feature, dev_label, dev_subsample, random_state
    )

    train_x = train_feature_sub.to_numpy()
    dev_x = dev_feature_sub.to_numpy()

    class_values, train_label_encoded = np.unique(train_label_sub.to_numpy(), return_inverse=True)
    dev_label_encoded = _encode_labels(dev_label_sub.to_numpy(), class_values)

    print(
        f"Grid search on {train_x.shape[0]:,} train / {dev_x.shape[0]:,} dev rows, "
        f"{len(class_values)} classes"
    )

    max_k = max(list_k)

    combinations = []
    for k_neighbors in list_k:
        for weight in weights:
            combinations.append((k_neighbors, weight))

    result_rows = []

    for metric in metrics:
        print(f"  metric={metric}: fitting NearestNeighbors at k={max_k}...")
        index = get_nearest_neighbors(train_x, metric, max_k)

        neighbor_distances, neighbor_indices = _query_neighbors_in_chunks(
            index, dev_x, query_chunk
        )

        with parallel_config(backend="threading"):
            scored_rows = Parallel(n_jobs=n_jobs, verbose=0)(
                delayed(_score_hyperparameter_combo)(
                    neighbor_indices,
                    neighbor_distances,
                    train_label_encoded,
                    dev_label_encoded,
                    k_neighbors,
                    weight,
                    metric,
                    len(class_values),
                )
                for k_neighbors, weight in combinations
            )

        result_rows.extend(scored_rows)

    result = pd.DataFrame(result_rows)
    result = result.sort_values("macro_f1", ascending=False, ignore_index=True)
    result.attrs["class_names"] = [str(c) for c in class_values]
    return result

In [57]:
def get_best_hyperparameter(result: pd.DataFrame,sort_by:str) -> tuple[int, str, str]:
    """Return the (k_neighbors, weight, metric) of the highest macro-F1 row.

    Selection stays on macro F1 (rare attack classes count as much as Benign);
    the weighted F1 is printed alongside only for context.
    """
    ranked = result.sort_values(sort_by, ascending=False)
    best_row = ranked.iloc[0]

    k_neighbors = int(best_row["k_neighbors"])
    weight = str(best_row["weight"])
    metric = str(best_row["metric"])

    print(
        f"Best: k={k_neighbors}, weight={weight}, metric={metric} "
        f"(macro F1 = {best_row['macro_f1']:.4f}, "
        f"weighted F1 = {best_row['weighted_f1']:.4f}, "
        f"accuracy = {best_row['accuracy']:.4f})"
    )
    return k_neighbors, weight, metric

In [45]:
results = knn_hyperparameter_grid_search(train_x,train_y,dev_x,dev_y,list_k_neighbors,list_weight, list_metric,KNN_TRAIN_SUBSAMPLE, KNN_DEV_SUBSAMPLE)

Grid search on 999,999 train / 199,999 dev rows, 15 classes
  metric=euclidean: fitting NearestNeighbors at k=15...
  metric=manhattan: fitting NearestNeighbors at k=15...


In [53]:
summary_columns = [
    "k_neighbors", "weight", "metric", "accuracy",
    "macro_precision", "macro_recall", "macro_f1",
    "weighted_precision", "weighted_recall", "weighted_f1",
]
results.sort_values("weighted_f1", ascending=False)[summary_columns].head(10)

,k_neighbors,weight,metric,accuracy,macro_precision,macro_recall,macro_f1,weighted_precision,weighted_recall,weighted_f1
21,13,uniform,manhattan,0.983270,0.737549,0.901315,0.751242,0.979401,0.983270,0.980330
19,11,uniform,euclidean,0.983140,0.736870,0.901318,0.752420,0.979238,0.983140,0.980306
23,13,uniform,euclidean,0.983210,0.736764,0.901147,0.750580,0.979285,0.983210,0.980274
24,15,uniform,euclidean,0.983265,0.736209,0.900937,0.749896,0.979308,0.983265,0.980269
27,15,uniform,manhattan,0.983230,0.734255,0.900804,0.747155,0.979284,0.983230,0.980236
16,9,uniform,euclidean,0.982880,0.737732,0.901628,0.755442,0.978992,0.982880,0.980140
18,7,uniform,euclidean,0.982240,0.733812,0.899646,0.752929,0.978390,0.982240,0.979587
17,5,uniform,euclidean,0.981915,0.734301,0.900288,0.754404,0.978321,0.981915,0.979442
14,15,distance,manhattan,0.981385,0.732917,0.902310,0.757051,0.978221,0.981385,0.979301
13,15,distance,euclidean,0.981410,0.735407,0.901982,0.759928,0.978143,0.981410,0.979268


#### Per-class breakdown

`macro_f1` averages every class equally, so a low value just means some classes
are weak -- it does not say which. `knn_per_class_report` expands the per-class
precision/recall/F1/support vectors carried on each result row (default: the
best macro-F1 row), sorted worst-F1 first, so the weak attack classes are
visible instead of being hidden inside the average.

In [54]:
def knn_per_class_report(results: pd.DataFrame,sort_by:str , rank: int = 0) -> pd.DataFrame:
    """Per-class precision/recall/F1/support for one row of the grid-search results.

    `rank` indexes into the macro-F1-sorted results (0 = best). Class names come
    from `results.attrs["class_names"]`, falling back to integer codes.
    """
    ranked = results.sort_values(sort_by, ascending=False, ignore_index=True)
    row = ranked.iloc[rank]

    class_names = results.attrs.get("class_names")
    if class_names is None:
        class_names = list(range(len(row["per_class_f1"])))

    report = pd.DataFrame({
        "class": class_names,
        "precision": row["per_class_precision"],
        "recall": row["per_class_recall"],
        "f1": row["per_class_f1"],
        "support": np.asarray(row["per_class_support"]).astype(int),
    }).sort_values("f1", ascending=True, ignore_index=True)

    print(
        f"k={int(row['k_neighbors'])}, weight={row['weight']}, metric={row['metric']} "
        f"| macro F1={row['macro_f1']:.4f}  weighted F1={row['weighted_f1']:.4f}  "
        f"accuracy={row['accuracy']:.4f}"
    )
    return report

In [55]:
knn_per_class_report(results,"weighted_f1", rank=0)

k=13, weight=uniform, metric=manhattan | macro F1=0.7512  weighted F1=0.9803  accuracy=0.9833


,class,precision,recall,f1,support
0,SQL Injection,0.052632,1.000000,0.100000,1
1,Infilteration,0.361426,0.111779,0.170750,1995
2,Brute Force -Web,0.160000,1.000000,0.275862,8
3,Brute Force -XSS,0.333333,1.000000,0.500000,3
4,DoS attacks-SlowHTTPTest,0.784062,0.530742,0.632999,1724
5,FTP-BruteForce,0.724830,0.894626,0.800827,2382
6,DDOS attack-LOIC-UDP,0.700000,1.000000,0.823529,21
7,DoS attacks-Slowloris,0.978102,0.992593,0.985294,135
8,DoS attacks-GoldenEye,0.984526,0.996086,0.990272,511
9,Benign,0.989369,0.997039,0.993189,166140


In [58]:
best_k, best_weight, best_metric = get_best_hyperparameter(results,"weighted_f1")

Best: k=13, weight=uniform, metric=manhattan (macro F1 = 0.7512, weighted F1 = 0.9803, accuracy = 0.9833)


### Train Model

In [ ]:
def to_float32(X) -> np.ndarray:
    """Contiguous float32 — halves memory traffic during distance computation."""
    return np.ascontiguousarray(X, dtype=np.float32)


def build_knn(k_neighbors: int, weight: str, metric: str,
              algorithm: str = "auto", n_jobs: int | None = -1) -> KNeighborsClassifier:
    """Config only, no data. Cheap to build many variants for tuning."""
    return KNeighborsClassifier(
        n_neighbors=k_neighbors,
        weights=weight,
        metric=metric,
        algorithm=algorithm,
        n_jobs=n_jobs,
    )


def fit_knn(model: KNeighborsClassifier, train_x, train_y) -> KNeighborsClassifier:
    return model.fit(to_float32(train_x), train_y)


def train_knn(train_x, train_y, k_neighbors: int, weight: str,
              metric: str, **kwargs) -> KNeighborsClassifier:
    """Thin orchestrator — keeps your existing call sites working."""
    return fit_knn(build_knn(k_neighbors, weight, metric, **kwargs), train_x, train_y)

In [60]:
def create_knn_model(
        train_x,
        train_y,
        k_neighbors:int,
        weight,
        metric:str) -> None :
    knn = train_knn(train_x,train_y,k_neighbors,weight,metric)
    dump_trained_model(knn,get_knn_name(k_neighbors,weight,metric),"KNN")

In [61]:
create_knn_model(train_x,train_y,best_k,best_weight,best_metric)

### Evaluate Model

The tuned KNN is reloaded from disk and scored **once** on the held-out test
split -- the split that was never touched during the hyperparameter search or
final training. Scoring goes through the same `evaluate_results` helper used
everywhere else, so the test numbers line up field-for-field with the dev
grid-search output above.

`test_y` still carries the real-world class balance (SMOTE only touched the
training split), so `macro_f1` -- every class weighted equally -- is the honest
headline, while the per-class table and confusion matrix below show which
attack classes actually pay for it.

In [41]:
def load_knn_model(model_path: str) -> KNeighborsClassifier:
    return joblib.load(os.path.join(PATH_FOLDER_MODEL, "KNN", model_path))

In [42]:
def knn_predict(model_path: str, test_feature: pl.DataFrame, query_chunk: int = KNN_QUERY_CHUNK) -> np.ndarray:
    """Predict a label for every test row, querying the fitted KNN in chunks.

    A brute-force KNN scores each query against the whole training set, so the
    test split is pushed through `query_chunk` rows at a time -- the same
    streaming idea as `_query_neighbors_in_chunks` in the grid search -- to
    bound peak memory and give progress feedback on what is a long call.
    """
    knn = load_knn_model(model_path)

    feature_matrix = test_feature.to_numpy()
    total_rows = feature_matrix.shape[0]

    prediction_chunks = []
    for start in range(0, total_rows, query_chunk):
        stop = min(start + query_chunk, total_rows)
        print(f"Predicting [{stop:,}/{total_rows:,}] test rows...", end="\r", flush=True)
        prediction_chunks.append(knn.predict(feature_matrix[start:stop]))

    print(" " * 100, end="\r")
    return np.concatenate(prediction_chunks)

In [44]:
knn_model_name = "KNN-k-13-w-uniform-m-manhattan.pkl"
test_predictions = knn_predict(knn_model_name, test_x)

C:\Users\JoshuaNs\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
  warnings.warn(


KeyboardInterrupt: 

In [ ]:
knn_test_report = evaluate_results(test_y.to_numpy(), test_predictions)
per_class_report(knn_test_report)

#### Confusion matrix

In [ ]:
plot_confusion_matrix(knn_test_report, normalize=True)